## 1. Препроцессинг NLP

**NLP препроцессинг** – это набор методов и техник для подготовки текста к анализу и обработке с использованием моделей машинного обучения и искусственного интеллекта. Этот этап включает несколько ключевых шагов, каждый из которых может включать различные подходы и инструменты.


### Основные этапы препроцессинга

#### 1. **Токенизация**
   Токенизация – процесс разделения текста на отдельные единицы анализа, такие как слова, символы или подстроки. Это важный шаг, так как большинство моделей работают именно с токенами.
   
   * **Методы токенизации:**
     - Простая токенизация по пробелам.
     - Регулярные выражения для учета знаков пунктуации.
     - Разбиение на лексемы (например, использование библиотек типа `nltk`).
     - Подтокенизация (например, при использовании трансформеров, таких как BERT, где длинные слова разбиваются на субтоксены).
     
   * **Модели токенизаторов:**
     - `WordPiece`: модель, используемая в BERT, которая разбивает слова на части, основываясь на частоте встречаемости частей слов.
     - `Byte-Pair Encoding (BPE)`: алгоритм, который сначала разбивает текст на байты, а затем объединяет часто встречающиеся последовательности символов в новые токены.
     - `SentencePiece`: еще одна популярная техника, которая комбинирует идеи WordPiece и BPE, но работает непосредственно с символами.


## 1. WordPiece Tokenization (используя Hugging Face Transformers)

In [ ]:
from transformers import BertTokenizer

'''Загружаем предобученную модель BERT tokenizer'''
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

'''Пример текста'''
text = "This is a test sentence for WordPiece tokenization."

'''Токенизация текста'''
tokens = tokenizer.tokenize(text)
print("Tokens:", tokens)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Tokens: ['this', 'is', 'a', 'test', 'sentence', 'for', 'word', '##piece', 'token', '##ization', '.']


## 2. Byte Pair Encoding (BPE) (используя SentencePiece)

In [ ]:
import sentencepiece as spm

'''Создаём список строк для обучения модели'''
sentences = [
    "This is the first sentence.",
    "This is the second sentence.",
    "And this is the third one."
]

'''Объединяем все предложения в одну строку'''
train_data = "\n".join(sentences)

'''Сохраняем данные в файл'''
with open("train.txt", "w") as f:
    f.write(train_data)

'''Параметры для обучения модели'''
params = {
    'model_prefix': 'bpe',  # Имя модели будет bpe.model
    'vocab_size': 25,       # Размер словаря
    'character_coverage': 1.0,
    'input_sentence_size': 5000000,
    'shuffle_input_sentence': True,
}

'''Обучаем модель'''
spm.SentencePieceTrainer.Train(
    f'--input=train.txt '
    f'--model_prefix={params["model_prefix"]} '
    f'--vocab_size={params["vocab_size"]} '
    f'--character_coverage={params["character_coverage"]} '
    f'--input_sentence_size={params["input_sentence_size"]} '
    f'--shuffle_input_sentence'
)

'''Загружаем созданную модель'''
spm_model = spm.SentencePieceProcessor()
spm_model.Load(f"{params['model_prefix']}.model")

'''Пример текста'''
text = "This is another test sentence for BPE encoding."

'''Токенизация текста'''
tokens = spm_model.EncodeAsPieces(text)
print("Tokens:", tokens)

Tokens: ['▁', 'T', 'hi', 's', '▁', 'i', 's', '▁', 'a', 'n', 'o', 't', 'h', 'e', 'r', '▁', 't', 'e', 's', 't', '▁se', 'n', 't', 'en', 'c', 'e', '▁', 'f', 'o', 'r', '▁', 'BPE', '▁', 'en', 'c', 'o', 'd', 'i', 'n', 'g', '.']


## 3. SentencePiece (комбинация WordPiece и BPE)

In [ ]:
!pip install sentencepiece

In [ ]:
import sentencepiece as spm

'''Шаг 1: Подготовка данных'''
input_file = '/content/1.txt'  # Файл с текстом для обучения модели

'''Шаг 2: Обучение модели с меньшим vocab_size'''
spm.SentencePieceTrainer.Train(f'--input={input_file} --model_prefix=m --vocab_size=45')

'''Шаг 3: Загрузка модели'''
sp = spm.SentencePieceProcessor(model_file='m.model')

'''Шаг 4: Токенизация текста'''
text = "Пример текста для токенизации."
tokens = sp.encode(text, out_type=str)
print("Токены:", tokens)

'''Шаг 5: Декодирование токенов обратно в текст'''
decoded_text = sp.decode(tokens)
print("Декодированный текст:", decoded_text)

Токены: ['▁', 'П', 'р', 'и', 'м', 'е', 'р', '▁текст', 'а', '▁', 'д', 'л', 'я', '▁', 'то', 'к', 'ен', 'и', 'з', 'а', 'ц', 'и', 'и', '.']
Декодированный текст: Пример текста для токенизации.


---------------------------------------------------

## 2. Удаление стоп-слов

Стоп-слова – это общие слова, которые встречаются очень часто и могут не нести смысловой нагрузки (например, союзы, предлоги). Их удаление помогает уменьшить размер данных и улучшить качество анализа.

    Подходы:
        Использование готовых списков стоп-слов (например, от библиотеки nlp или spacy).
        Создание собственных списков на основе частоты использования слов в конкретном корпусе текстов.
        Удаление наиболее частотных слов после анализа распределения частот.


In [ ]:
!pip install nltk

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import nltk

nltk.download('punkt')
nltk.download('stopwords')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

text = """
Это пример текста, который мы будем использовать для удаления стоп-слов.
Стоп-слова часто не несут смысловой нагрузки и могут быть удалены.
"""

tokens = word_tokenize(text)

stop_words = set(stopwords.words('russian'))
filtered_tokens = [word for word in tokens if word.lower() not in stop_words]

print("Токены без стоп-слов:", filtered_tokens)

Токены без стоп-слов: ['Это', 'пример', 'текста', ',', 'который', 'будем', 'использовать', 'удаления', 'стоп-слов', '.', 'Стоп-слова', 'часто', 'несут', 'смысловой', 'нагрузки', 'могут', 'удалены', '.']


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


----------------------------------------------------------------

## 3. **Лемматизация и стемминг**
   Эти процессы помогают нормализовать формы слов, приводя их к базовой форме.
   
   * **Стемминг**: упрощенная версия нормализации, когда слово обрезается до корня без учета грамматических правил. Например, "walked" → "walk".
     - Популярным инструментом является алгоритм Портера (`Porter Stemmer`), хотя существуют и другие алгоритмы (например, Snowball).
   
   * **Лемматизация**: более сложный подход, который учитывает морфологию языка и приводит слово к его лемме (базовая форма). Например, "was" → "be". Лемматизаторы используют словарь и правила склонения/спряжения.
     - Примеры инструментов: `WordNetLemmatizer`, встроенный в `nltk`.


In [ ]:
import nltk
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

'''Загрузка необходимых ресурсов'''
nltk.download('wordnet')
nltk.download('punkt')

'''Инициализация стеммера и лемматизатора'''
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

'''Пример текста для анализа'''
text = """
The cats were walking and jumped over the fence.
The dogs are running quickly.
"""

'''Токенизация текста'''
tokens = nltk.word_tokenize(text)

'''Стемминг'''
stemmed_words = [stemmer.stem(word) for word in tokens]
print("Стемминг:", stemmed_words)

'''Лемматизация'''
lemmatized_words = [lemmatizer.lemmatize(word) for word in tokens]
print("Лемматизация:", lemmatized_words)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Стемминг: ['the', 'cat', 'were', 'walk', 'and', 'jump', 'over', 'the', 'fenc', '.', 'the', 'dog', 'are', 'run', 'quickli', '.']
Лемматизация: ['The', 'cat', 'were', 'walking', 'and', 'jumped', 'over', 'the', 'fence', '.', 'The', 'dog', 'are', 'running', 'quickly', '.']


--------------------------------------------------------

4. POS-tagging

Позволяет определить часть речи каждого слова в предложении (существительное, глагол, прилагательное и т.д.). Это полезно для синтаксического анализа и понимания структуры предложения.

    Инструменты:
        SpaCy: библиотека с предобученными моделями POS-теггирования.
        Stanford CoreNLP: мощный инструмент для различных задач NLP, включая POS-теггирование.
        TreeTagger: популярный теггер, поддерживающий множество языков.


In [ ]:
!pip install spacy >> None

In [ ]:
!python -m spacy download en_core_web_sm >> None

In [ ]:
import spacy

'''Загрузка предобученной модели'''
nlp = spacy.load("en_core_web_sm")

'''Пример текста для анализа'''
text = "The quick brown fox jumps over the lazy dog."

'''Обработка текста'''
doc = nlp(text)

'''Вывод частей речи для каждого слова'''
for token in doc:
    print(f"{token.text}: {token.pos_}")

The: DET
quick: ADJ
brown: ADJ
fox: NOUN
jumps: VERB
over: ADP
the: DET
lazy: ADJ
dog: NOUN
.: PUNCT


---------------------------------------------------

## 5. Corpus Pruning
   Иногда требуется удалить редко используемые слова или те, которые появляются слишком часто, чтобы снизить шум в данных.
   
   * **Методы:**
     - Минимальная частота: удаляются все слова, которые встречаются реже определенного порога.
     - Максимальная частота: удаляются слова, которые встречаются чаще заданной границы.


In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter

nltk.download('punkt')

text = """
The quick brown fox jumps over the lazy dog.
The dog barked and the fox ran away.
The fox is quick and the dog is lazy.
"""

'''Токенизация текста'''
tokens = word_tokenize(text)

'''Подсчет частоты слов'''
word_freq = Counter(tokens)

'''Определение порогов'''
min_freq = 2  # Минимальная частота
max_freq = 3  # Максимальная частота

'''Удаление редко и часто используемых слов'''
pruned_tokens = [
    word for word in tokens
    if word_freq[word] >= min_freq and word_freq[word] <= max_freq
]

'''Результаты'''
print("Исходные токены:", tokens)
print("Токены после удаления:", pruned_tokens)

Исходные токены: ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.', 'The', 'dog', 'barked', 'and', 'the', 'fox', 'ran', 'away', '.', 'The', 'fox', 'is', 'quick', 'and', 'the', 'dog', 'is', 'lazy', '.']
Токены после удаления: ['The', 'quick', 'fox', 'the', 'lazy', 'dog', '.', 'The', 'dog', 'and', 'the', 'fox', '.', 'The', 'fox', 'is', 'quick', 'and', 'the', 'dog', 'is', 'lazy', '.']


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


---------------------------------------------

6. Нейронный препроцессинг

Современные нейросетевые архитектуры, такие как трансформеры, требуют специальных подходов к препроцессингу, поскольку они работают с последовательностями фиксированной длины и используют специфические эмбеддинги.

- BERT-препроцессинг: включает в себя токенизацию по методу WordPiece, добавление специальных маркеров начала и конца предложений [CLS] и [SEP], а также маскировку некоторых токенов для задачи предсказания следующего слова.
- GPT-препроцессинг: требует лишь добавления специального маркера конца строки и корректную обработку длинных последовательностей.


In [ ]:
!pip install transformers >> None

In [ ]:
from transformers import BertTokenizer, GPT2Tokenizer

'''Пример текста для анализа'''
text = "Hello, how are you doing today?"

'''BERT-препроцессинг'''
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_inputs = bert_tokenizer(text,
                              add_special_tokens=True,  # Добавляем специальные токены [CLS] и [SEP]
                              padding='max_length',      # Добавляем паддинг до максимальной длины
                              truncation=True,           # Обрезаем до максимальной длины
                              return_tensors='pt')      # Возвращаем тензоры PyTorch

print("BERT токены:", bert_inputs['input_ids'])
print("BERT внимание маска:", bert_inputs['attention_mask'])

'''GPT-препроцессинг'''
gpt_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

'''Установка EOS токена как токена паддинга'''
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

gpt_inputs = gpt_tokenizer(text,
                            add_special_tokens=False,  # У GPT нет специальных токенов
                            padding='max_length',       # Добавляем паддинг до максимальной длины
                            truncation=True,            # Обрезаем до максимальной длины
                            return_tensors='pt')       # Возвращаем тензоры PyTorch

print("GPT токены:", gpt_inputs['input_ids'])

BERT токены: tensor([[ 101, 7592, 1010, 2129, 2024, 2017, 2725, 2651, 1029,  102,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0, 

----------------------------------------------------------

## 7. **Эмбеддинг слов (Word Embeddings)**
   Эмбеддинги представляют собой числовые представления слов, которые сохраняют семантику и контекст. Они используются для ввода слов в модели машинного обучения.
   
   * **Популярные эмбеддинги:**
     - `Word2Vec`: Модель, обучаемая на больших объемах текста, представляющая каждое слово в виде вектора. Есть два основных подхода: CBOW (Continuous Bag of Words) и Skip-Gram.
     - `FastText`: Расширение Word2Vec, которое использует субтокены для улучшения качества эмбеддинга редких слов.
     - `GloVe`: Еще одна популярная модель эмбеддинга, основанная на совместной вероятности появления слов в контексте.
     - `ELMo`: Контекстуализированные эмбеддинги, которые изменяют представление слова в зависимости от контекста.
     - `BERT`: Преобученная архитектура, которая создает контекстуализированные эмбеддинги на основе трансформеров.



In [ ]:
!pip install gensim >> None

Word2Vec

In [ ]:
import gensim
from gensim.models import Word2Vec

'''Пример предложений для обучения'''
sentences = [
    ["the", "cat", "sat", "on", "the", "mat"],
    ["the", "dog", "barked"],
    ["the", "cat", "and", "the", "dog", "played"],
    ["cats", "and", "dogs", "are", "animals"]
]

'''Обучение модели Word2Vec'''
model = Word2Vec(sentences, vector_size=50, window=2, min_count=1, sg=0)  # sg=0 для CBOW

'''Получение эмбеддинга для слова "cat"'''
cat_embedding = model.wv['cat']
print("Эмбеддинг для 'cat':", cat_embedding)

'''Поиск похожих слов'''
similar_words = model.wv.most_similar('cat', topn=5)
print("Похожие слова к 'cat':", similar_words)

Эмбеддинг для 'cat': [ 1.56351421e-02 -1.90203730e-02 -4.11062239e-04  6.93839323e-03
 -1.87794445e-03  1.67635437e-02  1.80215668e-02  1.30730132e-02
 -1.42324204e-03  1.54208085e-02 -1.70686692e-02  6.41421322e-03
 -9.27599426e-03 -1.01779103e-02  7.17923651e-03  1.07406788e-02
  1.55390287e-02 -1.15330126e-02  1.48667218e-02  1.32509926e-02
 -7.41960062e-03 -1.74912829e-02  1.08749345e-02  1.30195115e-02
 -1.57510047e-03 -1.34197120e-02 -1.41718509e-02 -4.99412045e-03
  1.02865072e-02 -7.33047491e-03 -1.87401194e-02  7.65347946e-03
  9.76895820e-03 -1.28571270e-02  2.41711619e-03 -4.14975407e-03
  4.88066689e-05 -1.97670180e-02  5.38400887e-03 -9.50021297e-03
  2.17529293e-03 -3.15244915e-03  4.39334614e-03 -1.57631524e-02
 -5.43436781e-03  5.32639725e-03  1.06933638e-02 -4.78302967e-03
 -1.90201886e-02  9.01175756e-03]
Похожие слова к 'cat': [('sat', 0.19010193645954132), ('cats', 0.0449172779917717), ('played', -0.010146019048988819), ('the', -0.014475265517830849), ('mat', -0.023

FastText

In [ ]:
from gensim.models import FastText

'''Обучение модели FastText'''
fasttext_model = FastText(sentences, vector_size=50, window=2, min_count=1, sg=1)

'''Получение эмбеддинга для слова "cat"'''
cat_embedding_fasttext = fasttext_model.wv['cat']
print("Эмбеддинг для 'cat' (FastText):", cat_embedding_fasttext)

'''Поиск похожих слов'''
similar_words_fasttext = fasttext_model.wv.most_similar('cat', topn=5)
print("Похожие слова к 'cat' (FastText):", similar_words_fasttext)

Эмбеддинг для 'cat' (FastText): [-0.00161995  0.00013274  0.00051012 -0.00034868  0.00488489  0.00073547
  0.00398982  0.00201859 -0.00160627 -0.00578359 -0.00434133 -0.00775934
 -0.00172438 -0.00369552  0.00505511  0.00149354 -0.00128937 -0.00467418
 -0.00090299  0.00509951 -0.00357978 -0.00210673  0.00743071  0.00635575
 -0.01105164 -0.00431297 -0.00216175 -0.00392401 -0.00865878 -0.00492574
 -0.00232304  0.00396143 -0.00018401 -0.00069397 -0.0013444  -0.00016649
  0.00203329 -0.00318087 -0.00297435 -0.00600852  0.00460726  0.00041121
  0.00649215 -0.00039054 -0.00562977  0.00194995 -0.00369214  0.00493267
 -0.00463476 -0.0006369 ]
Похожие слова к 'cat' (FastText): [('cats', 0.4415692687034607), ('and', 0.20802311599254608), ('mat', 0.18850363790988922), ('sat', 0.11652231961488724), ('dogs', 0.06727516651153564)]


BERT

In [ ]:
!pip install transformers >> None

In [ ]:
from transformers import BertTokenizer, BertModel
import torch

'''Инициализация модели и токенизатора BERT'''
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

'''Пример текста'''
text = "The cat sat on the mat."

'''Токенизация и получение индексированных токенов'''
inputs = tokenizer(text, return_tensors='pt')

'''Получение эмбеддингов'''
with torch.no_grad():
    outputs = model(**inputs)

'''Получение эмбеддинга для слова "cat"'''
cat_index = inputs['input_ids'][0].tolist().index(tokenizer.encode('cat')[1])
cat_embedding_bert = outputs.last_hidden_state[0][cat_index]
print("Эмбеддинг для 'cat' (BERT):", cat_embedding_bert)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Эмбеддинг для 'cat' (BERT): tensor([-3.5117e-01, -7.3560e-02, -6.9140e-02, -1.3988e-01,  6.8295e-01,
         1.1351e-01,  2.0849e-01,  5.6738e-01,  4.0695e-01, -2.1347e-01,
        -9.4565e-02, -7.2686e-01, -4.2637e-01, -1.6395e-01, -3.4623e-01,
        -3.9071e-01,  4.9144e-01,  2.4878e-01, -8.7979e-01,  1.1039e+00,
        -1.4416e-01, -8.7562e-01, -1.4432e+00, -2.6560e-01,  2.0460e-01,
        -5.8215e-02,  7.2639e-01,  1.8093e-01,  1.7829e-02,  4.5502e-02,
         3.1111e-01, -4.2371e-01,  1.1806e-01,  2.4286e-02, -1.4112e-01,
         5.4676e-02, -4.1218e-02,  4.2631e-01, -8.6915e-01,  3.7737e-02,
         6.4482e-01,  1.0988e-01, -6.8318e-02,  8.8491e-01,  7.6597e-01,
        -1.1312e-01,  3.5481e-01, -2.7578e-01,  1.5118e+00, -1.2058e-01,
        -3.4370e-01,  9.6826e-01, -6.7061e-01,  2.1608e-01, -3.0331e-02,
         3.5732e-01,  3.6955e-01, -7.1105e-01,  1.6722e-01,  6.2503e-01,
        -1.8112e-01,  4.7781e-01,  6.1709e-01, -6.7334e-01, -1.1165e+00,
         8.1133e-01, -4

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter

nltk.download('punkt')

text = """
The quick brown fox jumps over the lazy dog.
The dog barked and the fox ran away.
The fox is quick and the dog is lazy.
"""

tokens = word_tokenize(text)

'''Подсчет частоты слов'''
word_freq = Counter(tokens)

'''Определение порогов'''
min_freq = 2  # Минимальная частота
max_freq = 3  # Максимальная частота

'''Удаление редко и часто используемых слов'''
pruned_tokens = [
    word for word in tokens
    if word_freq[word] >= min_freq and word_freq[word] <= max_freq
]

'''Результаты'''
print("Исходные токены:", tokens)
print("Токены после удаления:", pruned_tokens)

Исходные токены: ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.', 'The', 'dog', 'barked', 'and', 'the', 'fox', 'ran', 'away', '.', 'The', 'fox', 'is', 'quick', 'and', 'the', 'dog', 'is', 'lazy', '.']
Токены после удаления: ['The', 'quick', 'fox', 'the', 'lazy', 'dog', '.', 'The', 'dog', 'and', 'the', 'fox', '.', 'The', 'fox', 'is', 'quick', 'and', 'the', 'dog', 'is', 'lazy', '.']


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
import gensim
from gensim.models import Word2Vec

'''Пример предложений для обучения'''
sentences = [
    ["the", "cat", "sat", "on", "the", "mat"],
    ["the", "dog", "barked"],
    ["the", "cat", "and", "the", "dog", "played"],
    ["cats", "and", "dogs", "are", "animals"]
]

'''Обучение модели Word2Vec'''
model = Word2Vec(sentences, vector_size=50, window=2, min_count=1, sg=0)  # sg=0 для CBOW

'''Получение эмбеддинга для слова "cat"'''
cat_embedding = model.wv['cat']
print("Эмбеддинг для 'cat':", cat_embedding)

'''Поиск похожих слов'''
similar_words = model.wv.most_similar('cat', topn=5)
print("Похожие слова к 'cat':", similar_words)

Эмбеддинг для 'cat': [ 1.56351421e-02 -1.90203730e-02 -4.11062239e-04  6.93839323e-03
 -1.87794445e-03  1.67635437e-02  1.80215668e-02  1.30730132e-02
 -1.42324204e-03  1.54208085e-02 -1.70686692e-02  6.41421322e-03
 -9.27599426e-03 -1.01779103e-02  7.17923651e-03  1.07406788e-02
  1.55390287e-02 -1.15330126e-02  1.48667218e-02  1.32509926e-02
 -7.41960062e-03 -1.74912829e-02  1.08749345e-02  1.30195115e-02
 -1.57510047e-03 -1.34197120e-02 -1.41718509e-02 -4.99412045e-03
  1.02865072e-02 -7.33047491e-03 -1.87401194e-02  7.65347946e-03
  9.76895820e-03 -1.28571270e-02  2.41711619e-03 -4.14975407e-03
  4.88066689e-05 -1.97670180e-02  5.38400887e-03 -9.50021297e-03
  2.17529293e-03 -3.15244915e-03  4.39334614e-03 -1.57631524e-02
 -5.43436781e-03  5.32639725e-03  1.06933638e-02 -4.78302967e-03
 -1.90201886e-02  9.01175756e-03]
Похожие слова к 'cat': [('sat', 0.19010193645954132), ('cats', 0.0449172779917717), ('played', -0.010146019048988819), ('the', -0.014475265517830849), ('mat', -0.023

----------------------------------------

### 8. **Очистка текста**
   Очистка текста включает удаление HTML-тегов, ссылок, эмодзи, лишних пробелов и других элементов, которые могут мешать анализу.
   
   * **Методы очистки:**
     - Регулярные выражения для удаления ненужных символов.
     - Специальные функции для удаления HTML-тегов (например, `BeautifulSoup` в Python).
     - Удаление эмодзи и специальных символов через Unicode-диапазоны.

In [ ]:
import re
from bs4 import BeautifulSoup
import string
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

def clean_text(text):
    '''Удаление HTML-тегов'''
    text = BeautifulSoup(text, "html.parser").get_text()

    '''Удаление ссылок'''
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    '''Удаление эмодзи'''
    text = re.sub(r'[^\x00-\x7F]+', '', text)

    '''Удаление знаков препинания'''
    text = text.translate(str.maketrans('', '', string.punctuation))

    '''Приведение к нижнему регистру'''
    text = text.lower()

    '''Удаление лишних пробелов'''
    text = re.sub(r'\s+', ' ', text).strip()

    '''Удаление цифр'''
    text = re.sub(r'\d+', '', text)

    '''Удаление стоп-слов'''
    stop_words = set(stopwords.words('english'))  # Использование стоп-слов из NLTK
    text = ' '.join([word for word in text.split() if word not in stop_words])

    '''Лемматизация'''
    lemmatizer = nltk.WordNetLemmatizer()
    text = ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

    return text

'''Пример текста'''
raw_text = "<p>This is a test sentence! Visit us at https://example.com 😊. It includes punctuation, links, numbers like 123, and more.</p>"

'''Очистка текста'''
cleaned_text = clean_text(raw_text)
print("Очищенный текст:", cleaned_text)

Очищенный текст: test sentence visit u includes punctuation link number like


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


------------------------------------------------

## 9. **Нормализация текста**
   Нормализация включает приведение всех букв к одному регистру (обычно нижнему), замену чисел на специальные токены и другие преобразования, делающие текст более однородным.
   
   * **Примеры нормализаций:**
     - Приведение к нижнему регистру.
     - Замена цифр на специальный токен `<NUM>` или аналогичное обозначение.
     - Унификация кодировок (например, замена всех вариантов кавычек на стандартные ASCII-коды).


In [ ]:
import re
import string
import unicodedata
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

'''Загрузка необходимых ресурсов'''
nltk.download('stopwords')
nltk.download('wordnet')

def normalize_text(text):
    '''Приведение к нижнему регистру'''
    text = text.lower()

    '''Удаление акцентов и диакритических знаков'''
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8')

    '''амена различных кавычек на стандартные ASCII'''
    text = text.replace('“', '"').replace('”', '"').replace('‘', "'").replace('’', "'")

    '''Замена цифр на специальный токен <NUM>'''
    text = re.sub(r'\d+', '<NUM>', text)

    '''Замена специальных символов (например, @, #, $, %, &, *) на пробел'''
    text = re.sub(r'[@#$%^&*]', ' ', text)

    '''Замена упоминаний на специальный токен <MENTION>'''
    text = re.sub(r'@\w+', '<MENTION>', text)

    '''Замена хештегов на специальный токен <HASHTAG>'''
    text = re.sub(r'#\w+', '<HASHTAG>', text)

    '''Удаление знаков препинания'''
    text = text.translate(str.maketrans('', '', string.punctuation))

    '''Удаление стоп-слов'''
    stop_words = set(stopwords.words('english'))
    text = ' '.join([word for word in text.split() if word not in stop_words])

    '''Лемматизация'''
    lemmatizer = WordNetLemmatizer()
    text = ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

    '''Удаление лишних пробелов'''
    text = re.sub(r'\s+', ' ', text).strip()

    '''Замена эмодзи на специальный токен <EMOJI>'''
    text = re.sub(r'[^\x00-\x7F]+', '<EMOJI>', text)

    '''Обработка сокращений: замена обычных сокращений на полные формы'''
    contractions = {
        "can't": "cannot",
        "won't": "will not",
        "don't": "do not",
        "it's": "it is",
        "he's": "he is",
        "she's": "she is",
        "they're": "they are",
        "we're": "we are",
        "i'm": "i am",
        "you're": "you are"
    }

    for contraction, full_form in contractions.items():
        text = text.replace(contraction, full_form)

    return text

'''Пример текста'''
raw_text = """This is a test sentence with numbers 123 and “quotes” that need to be normalized! Check @user for more info. Also, there's a price of $99.99 and a special character: é. #example Can't believe it's true! 😊"""

'''Нормализация текста'''
normalized_text = normalize_text(raw_text)
print("Нормализованный текст:", normalized_text)

Нормализованный текст: test sentence number NUM quote need normalized check user info also there price NUMNUM special character e example cant believe true


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Дополнительная литература

- https://habr.com/ru/articles/468141/
- https://habr.com/ru/companies/oleg-bunin/articles/352614/
- https://habr.com/ru/articles/738176/
- https://habr.com/ru/companies/getmeit/articles/579518/
- https://habr.com/ru/companies/Voximplant/articles/446738/
- https://habr.com/ru/articles/778048/
- https://habr.com/ru/articles/856436/
- https://habr.com/ru/companies/data_light/articles/857142/
- https://habr.com/ru/companies/bastion/articles/856180/
- https://habr.com/ru/companies/wunderfund/articles/859232/

- https://arxiv.org/pdf/2412.18296
- https://arxiv.org/pdf/2412.17347
- https://arxiv.org/pdf/2411.11171
- https://arxiv.org/pdf/2411.05026
- https://arxiv.org/pdf/2409.19013
- https://arxiv.org/pdf/2409.13920
- https://arxiv.org/pdf/2408.08694
- https://arxiv.org/pdf/2407.12376
- https://arxiv.org/pdf/2407.08328
- https://arxiv.org/pdf/2406.09765